# SciCite Direct-Input ToW Extensions: CC-ToW and IW-ToW

This notebook evaluates two direct-input adaptations of Thoughts of Words (ToW) for SciCite citation-intent classification. It does **not** generate ToW annotations and does **not** call the OpenAI API. Instead, it loads cached CC-ToW and IW-ToW files that were generated in earlier notebooks.

The evaluated conditions are:

1. **Raw SciBERT baseline**: the citation context only.
2. **Citation-Cue ToW (CC-ToW)**: a task-oriented ToW annotation that summarizes cue words or phrases relevant to the SciCite label, appended as an input feature.
3. **Inline Word-Level ToW (IW-ToW)**: a cleaned, word-level ToW annotation inserted directly into the citation context before a selected target word.

For a fair comparison, the notebook trains and evaluates all three conditions on the same common example IDs whenever possible. It saves tables, predictions, qualitative examples, and figures under:

```text
/content/drive/MyDrive/tow_project/B/extension/scicite/results_direct_input_tow_extensions
```

Run this notebook after the CC-ToW and IW-ToW datasets have already been generated.


In [ ]:
# ============================================================
# 1. Mount Drive and define paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, re, random, inspect, warnings
from collections import Counter, defaultdict

PROJECT_ROOT = Path('/content/drive/MyDrive/tow_project/B/extension/scicite')
DATA_DIR = PROJECT_ROOT / 'data'
TASK_TOW_DIR = PROJECT_ROOT / 'tow_generations'                 # CC-ToW generated annotations
WORDLEVEL_TOW_DIR = PROJECT_ROOT / 'wordlevel_tow_generations_clean_v2'  # clean IW-ToW annotations
RESULTS_DIR = PROJECT_ROOT / 'results_direct_input_tow_extensions'
FIG_DIR = RESULTS_DIR / 'figures'
PRED_DIR = RESULTS_DIR / 'predictions'

for p in [RESULTS_DIR, FIG_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('TASK_TOW_DIR:', TASK_TOW_DIR)
print('WORDLEVEL_TOW_DIR:', WORDLEVEL_TOW_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

In [ ]:
# ============================================================
# 2. Runtime and experiment settings
# ============================================================
SEED = 42
random.seed(SEED)
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Model for the extension classifier.
# SciBERT is the better paper setting for scientific text; change to
# 'distilbert-base-uncased' if you want to reproduce the earlier quick run.
CLASSIFIER_MODEL = 'allenai/scibert_scivocab_uncased'

MAX_LENGTH = 256
NUM_EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Important for fair comparison:
# True = Raw, CC-ToW, and IW-ToW are all trained/evaluated on the same IDs per split.
# This avoids giving one condition more data than another.
USE_COMMON_IDS_FOR_ALL_CONDITIONS = True

# The IW-ToW generator is expected to keep useful exact/soft-consistent rows.
# This remains as a safety filter for reproducible evaluation.
B_ALLOWED_CATEGORIES = {'exact_match', 'soft_consistent'}

# Rerun training even if existing metrics are present.
# Set to False only when intentionally reusing cached outputs.
FORCE_RETRAIN = True

print('CLASSIFIER_MODEL:', CLASSIFIER_MODEL)
print('NUM_EPOCHS:', NUM_EPOCHS)
print('USE_COMMON_IDS_FOR_ALL_CONDITIONS:', USE_COMMON_IDS_FOR_ALL_CONDITIONS)
print('FORCE_RETRAIN:', FORCE_RETRAIN)

In [ ]:
# ============================================================
# 3. Install/import only testing dependencies
# ============================================================
import sys, subprocess, importlib.util

required_imports = {
    'transformers': 'transformers',
    'datasets': 'datasets',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'accelerate': 'accelerate',
}
missing = [pip_name for import_name, pip_name in required_imports.items() if importlib.util.find_spec(import_name) is None]

if missing:
    print('Installing missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('All required packages already importable; skipping pip install.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('torch:', torch.__version__)
try:
    import transformers, datasets
    print('transformers:', transformers.__version__)
    print('datasets:', datasets.__version__)
except Exception as e:
    print('version check warning:', repr(e))
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 4. JSONL helpers and validators
# ============================================================
def read_jsonl(path):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)

def write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    tmp.replace(path)

def find_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

def raw_path(split):
    return find_existing_path([
        DATA_DIR / f'scicite_{split}_raw_sample.jsonl',
        DATA_DIR / f'scicite_{split}_raw.jsonl',
        PROJECT_ROOT / f'scicite_{split}_raw_sample.jsonl',
    ])

def task_tow_path(split):
    return find_existing_path([
        TASK_TOW_DIR / f'scicite_{split}_tow.jsonl',
        PROJECT_ROOT / 'tow_generations' / f'scicite_{split}_tow.jsonl',
    ])

def wordlevel_tow_path(split):
    return find_existing_path([
        WORDLEVEL_TOW_DIR / f'scicite_{split}_wordlevel_tow.jsonl',
        PROJECT_ROOT / 'wordlevel_tow_generations_clean_v2' / f'scicite_{split}_wordlevel_tow.jsonl',
        PROJECT_ROOT / 'wordlevel_tow_generations_clean' / f'scicite_{split}_wordlevel_tow.jsonl',
        PROJECT_ROOT / 'wordlevel_tow_generations' / f'scicite_{split}_wordlevel_tow.jsonl',
    ])

def validate_raw_row(r):
    return (
        isinstance(r, dict)
        and isinstance(r.get('id'), str)
        and isinstance(r.get('text'), str)
        and len(r.get('text', '').strip()) > 0
        and r.get('label') is not None
    )

def validate_task_tow_row(r):
    return (
        isinstance(r, dict)
        and isinstance(r.get('id'), str)
        and isinstance(r.get('text'), str)
        and r.get('label') is not None
        and isinstance(r.get('tow'), str)
        and len(r.get('tow', '').strip()) > 20
    )

def validate_wordlevel_tow_row(r):
    if not (
        isinstance(r, dict)
        and isinstance(r.get('id'), str)
        and isinstance(r.get('text'), str)
        and r.get('label') is not None
        and isinstance(r.get('target_word'), str)
        and len(r.get('target_word', '').strip()) > 0
        and isinstance(r.get('tow_augmented_text'), str)
        and '<ToW>' in r.get('tow_augmented_text', '')
        and '</ToW>' in r.get('tow_augmented_text', '')
    ):
        return False
    cat = r.get('tow_category')
    if cat is not None and B_ALLOWED_CATEGORIES and cat not in B_ALLOWED_CATEGORIES:
        return False
    return True

print('Helper functions loaded.')

In [ ]:
# ============================================================
# 5. Load raw, Citation-Cue ToW (CC-ToW), and clean Inline Word-Level ToW (IW-ToW) files
# ============================================================
splits = ['train', 'validation', 'test']
raw_rows = {}
task_rows = {}
word_rows = {}

for split in splits:
    rp = raw_path(split)
    ap = task_tow_path(split)
    bp = wordlevel_tow_path(split)
    print('\n---', split, '---')
    print('raw path:', rp)
    print('CC-ToW path:', ap)
    print('IW-ToW path:', bp)
    if rp is None:
        raise FileNotFoundError(f'Missing raw file for split={split}')
    if ap is None:
        raise FileNotFoundError(f'Missing Citation-Cue ToW (CC-ToW) task-ToW file for split={split}')
    if bp is None:
        raise FileNotFoundError(f'Missing clean Inline Word-Level ToW (IW-ToW) word-level ToW file for split={split}')

    raw_rows[split] = [r for r in read_jsonl(rp) if validate_raw_row(r)]
    task_rows[split] = [r for r in read_jsonl(ap) if validate_task_tow_row(r)]
    word_rows[split] = [r for r in read_jsonl(bp) if validate_wordlevel_tow_row(r)]

    print('raw valid:', len(raw_rows[split]))
    print('CC-ToW valid:', len(task_rows[split]))
    print('IW-ToW valid:', len(word_rows[split]))
    print('IW-ToW categories:', Counter(r.get('tow_category') for r in word_rows[split]))

# Make label-name lookup.
all_raw = raw_rows['train'] + raw_rows['validation'] + raw_rows['test']
label_to_names = defaultdict(Counter)
for r in all_raw:
    if r.get('label_name') is not None:
        label_to_names[int(r['label'])][str(r['label_name'])] += 1

label_values = sorted({int(r['label']) for r in all_raw})
LABEL_NAMES = []
for lab in label_values:
    if label_to_names[lab]:
        LABEL_NAMES.append(label_to_names[lab].most_common(1)[0][0])
    else:
        LABEL_NAMES.append(str(lab))

id2label = {i: LABEL_NAMES[i] if i < len(LABEL_NAMES) else str(i) for i in label_values}
label2id = {v: k for k, v in id2label.items()}
NUM_LABELS = len(label_values)

print('\nLABEL_NAMES:', LABEL_NAMES)
print('NUM_LABELS:', NUM_LABELS)

In [ ]:
# ============================================================
# 6. Build fair direct-input classifier datasets from cached CC-ToW and IW-ToW files
# ============================================================
def map_by_id(rows):
    return {r['id']: r for r in rows}

def get_common_ids_for_split(split):
    raw_ids = set(map_by_id(raw_rows[split]))
    a_ids = set(map_by_id(task_rows[split]))
    b_ids = set(map_by_id(word_rows[split]))
    return sorted(raw_ids & a_ids & b_ids)

common_ids = {split: get_common_ids_for_split(split) for split in splits}
print('Common IDs across raw/CC/IW:', {s: len(common_ids[s]) for s in splits})

if any(len(common_ids[s]) == 0 for s in splits):
    raise RuntimeError('At least one split has zero common IDs. Check file paths and ID names.')

# For paper fairness, default to common ID subset for every condition.
selected_ids = common_ids if USE_COMMON_IDS_FOR_ALL_CONDITIONS else None

condition_descriptions = {
    'baseline_raw': 'Raw citation context only',
    'citation_cue_tow': 'Citation-Cue ToW (CC-ToW): citation-cue/task-oriented ToW appended as an input feature',
    'inline_wordlevel_tow': 'Inline Word-Level ToW (IW-ToW): clean word-level ToW inserted into citation context',
}

def build_examples(condition, split):
    raw_map = map_by_id(raw_rows[split])
    task_map = map_by_id(task_rows[split])
    word_map = map_by_id(word_rows[split])

    if selected_ids is not None:
        ids = selected_ids[split]
    else:
        if condition == 'baseline_raw':
            ids = sorted(raw_map)
        elif condition == 'citation_cue_tow':
            ids = sorted(set(raw_map) & set(task_map))
        elif condition == 'inline_wordlevel_tow':
            ids = sorted(set(raw_map) & set(word_map))
        else:
            raise ValueError(condition)

    examples = []
    for ex_id in ids:
        r = raw_map[ex_id]
        label = int(r['label'])
        label_name = r.get('label_name')

        if condition == 'baseline_raw':
            text = f"Citation: {r['text']}"
        elif condition == 'citation_cue_tow':
            a = task_map[ex_id]
            text = f"Citation: {a['text']}\nThought of words: {a['tow']}"
        elif condition == 'inline_wordlevel_tow':
            b = word_map[ex_id]
            text = f"Citation: {b['tow_augmented_text']}"
        else:
            raise ValueError(condition)

        examples.append({
            'id': ex_id,
            'text': text,
            'label': label,
            'label_name': label_name,
        })
    return examples

def build_datasetdict(condition):
    return DatasetDict({split: Dataset.from_list(build_examples(condition, split)) for split in splits})

datasets_by_condition = {cond: build_datasetdict(cond) for cond in condition_descriptions}

for cond, dd in datasets_by_condition.items():
    print('\n', cond)
    print(dd)
    print('example:', dd['train'][0]['text'][:500])
    print('label:', dd['train'][0]['label'], dd['train'][0].get('label_name'))

# Save dataset audit.
audit = {
    'classifier_model': CLASSIFIER_MODEL,
    'use_common_ids_for_all_conditions': USE_COMMON_IDS_FOR_ALL_CONDITIONS,
    'common_id_counts': {k: len(v) for k, v in common_ids.items()},
    'conditions': condition_descriptions,
    'dataset_sizes': {cond: {s: len(dd[s]) for s in splits} for cond, dd in datasets_by_condition.items()},
    'label_names': LABEL_NAMES,
    'paths': {
        'data_dir': str(DATA_DIR),
        'task_tow_dir': str(TASK_TOW_DIR),
        'wordlevel_tow_dir': str(WORDLEVEL_TOW_DIR),
        'results_dir': str(RESULTS_DIR),
    },
}
write_json(audit, RESULTS_DIR / 'dataset_audit.json')
print('\nSaved dataset audit:', RESULTS_DIR / 'dataset_audit.json')

In [ ]:
# ============================================================
# 7. Train/evaluate a classifier for Raw, Citation-Cue ToW (CC-ToW), Inline Word-Level ToW (IW-ToW)
# ============================================================
def make_training_args(output_dir):
    base_kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=10,
        save_strategy='no',
        report_to=[],
        seed=SEED,
        data_seed=SEED,
    )

    # Transformers changed evaluation_strategy -> eval_strategy in newer versions.
    sig = inspect.signature(TrainingArguments.__init__)
    if 'evaluation_strategy' in sig.parameters:
        base_kwargs['evaluation_strategy'] = 'epoch'
    elif 'eval_strategy' in sig.parameters:
        base_kwargs['eval_strategy'] = 'epoch'

    return TrainingArguments(**base_kwargs)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

def safe_trainer(model, args, train_dataset, eval_dataset, tokenizer, data_collator):
    kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    try:
        return Trainer(**kwargs, tokenizer=tokenizer)
    except TypeError:
        # Newer Transformers may reject tokenizer=...
        return Trainer(**kwargs)

def train_and_evaluate(condition, dd):
    metrics_path = RESULTS_DIR / f'{condition}_metrics.json'
    report_path = RESULTS_DIR / f'{condition}_classification_report.json'
    preds_path = PRED_DIR / f'{condition}_test_predictions.jsonl'
    cm_path = RESULTS_DIR / f'{condition}_confusion_matrix.json'

    if metrics_path.exists() and not FORCE_RETRAIN:
        print(f'\n=== Reusing cached metrics for {condition} ===')
        return json.load(open(metrics_path, 'r', encoding='utf-8'))

    print(f'\n=== Training/evaluating {condition} ===')
    tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def tok(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

    encoded = dd.map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL,
        num_labels=NUM_LABELS,
        id2label={i: LABEL_NAMES[i] for i in range(NUM_LABELS)},
        label2id={LABEL_NAMES[i]: i for i in range(NUM_LABELS)},
    )

    args = make_training_args(RESULTS_DIR / 'trainer_runs' / condition)
    trainer = safe_trainer(
        model=model,
        args=args,
        train_dataset=encoded['train'],
        eval_dataset=encoded['validation'],
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    pred_out = trainer.predict(encoded['test'])
    logits = pred_out.predictions
    labels = pred_out.label_ids
    preds = np.argmax(logits, axis=-1)

    acc = float(accuracy_score(labels, preds))
    macro_f1 = float(f1_score(labels, preds, average='macro'))
    per_class_f1 = f1_score(labels, preds, average=None, labels=list(range(NUM_LABELS)))
    report = classification_report(
        labels,
        preds,
        labels=list(range(NUM_LABELS)),
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_LABELS))).tolist()

    rows = []
    ids = dd['test']['id']
    texts = dd['test']['text']
    label_names = dd['test']['label_name'] if 'label_name' in dd['test'].column_names else [None] * len(ids)
    for i, ex_id in enumerate(ids):
        rows.append({
            'id': ex_id,
            'gold_label': int(labels[i]),
            'gold_label_name': LABEL_NAMES[int(labels[i])] if int(labels[i]) < len(LABEL_NAMES) else str(labels[i]),
            'pred_label': int(preds[i]),
            'pred_label_name': LABEL_NAMES[int(preds[i])] if int(preds[i]) < len(LABEL_NAMES) else str(preds[i]),
            'correct': bool(labels[i] == preds[i]),
            'text': texts[i],
        })

    metrics = {
        'condition': condition,
        'description': condition_descriptions[condition],
        'test_accuracy': acc,
        'test_macro_f1': macro_f1,
        'num_train': len(dd['train']),
        'num_validation': len(dd['validation']),
        'num_test': len(dd['test']),
        'classifier_model': CLASSIFIER_MODEL,
        'num_epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH,
        'use_common_ids_for_all_conditions': USE_COMMON_IDS_FOR_ALL_CONDITIONS,
    }
    for i, name in enumerate(LABEL_NAMES):
        metrics[f'f1_{name}'] = float(per_class_f1[i])

    write_json(metrics, metrics_path)
    write_json(report, report_path)
    write_json({'labels': LABEL_NAMES, 'matrix': cm}, cm_path)
    write_jsonl(rows, preds_path)

    print('Saved metrics:', metrics_path)
    print('Saved predictions:', preds_path)
    print(metrics)

    # Free memory between conditions.
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

results = []
for condition, dd in datasets_by_condition.items():
    results.append(train_and_evaluate(condition, dd))

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_DIR / 'scicite_direct_input_tow_results.csv', index=False)
write_json(results, RESULTS_DIR / 'scicite_direct_input_tow_results.json')
print('\nSaved final results:')
print(RESULTS_DIR / 'scicite_direct_input_tow_results.csv')
display(results_df)

In [ ]:
# ============================================================
# 8. Tables, deltas, and figures
# ============================================================
results_df = pd.read_csv(RESULTS_DIR / 'scicite_direct_input_tow_results.csv')

# Stable order for display.
order = ['baseline_raw', 'citation_cue_tow', 'inline_wordlevel_tow']
results_df['order'] = results_df['condition'].apply(lambda x: order.index(x) if x in order else 99)
results_df = results_df.sort_values('order').drop(columns=['order'])

raw_row = results_df[results_df['condition'] == 'baseline_raw'].iloc[0]
results_df['delta_accuracy_vs_raw'] = results_df['test_accuracy'] - raw_row['test_accuracy']
results_df['delta_macro_f1_vs_raw'] = results_df['test_macro_f1'] - raw_row['test_macro_f1']

report_cols = [
    'condition', 'test_accuracy', 'test_macro_f1',
    'delta_accuracy_vs_raw', 'delta_macro_f1_vs_raw',
    'num_train', 'num_validation', 'num_test',
]
# Include per-class F1 columns if present.
for c in results_df.columns:
    if c.startswith('f1_'):
        report_cols.append(c)

final_table = results_df[report_cols].copy()
final_table_path = RESULTS_DIR / 'scicite_direct_input_tow_results_paper_table.csv'
final_table.to_csv(final_table_path, index=False)
write_json(final_table.to_dict(orient='records'), RESULTS_DIR / 'scicite_direct_input_tow_results_paper_table.json')

print('Saved paper table:', final_table_path)
display(final_table)

# Main bar chart.
plot_df = results_df.copy()
plot_df['label'] = plot_df['condition'].map({
    'baseline_raw': 'Raw baseline',
    'citation_cue_tow': 'Ext. A\nTask ToW',
    'inline_wordlevel_tow': 'Ext. B\nWord-level ToW',
}).fillna(plot_df['condition'])

x = np.arange(len(plot_df))
width = 0.36
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.bar(x - width/2, plot_df['test_accuracy'], width, label='Accuracy')
ax.bar(x + width/2, plot_df['test_macro_f1'], width, label='Macro F1')
ax.set_ylabel('Score')
ax.set_title('SciCite Direct-Input ToW Results')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['label'])
ax.set_ylim(0, 1.0)
ax.legend(loc='lower right')
for i, row in plot_df.iterrows():
    idx = list(plot_df.index).index(i)
    ax.text(idx - width/2, row['test_accuracy'] + 0.015, f"{row['test_accuracy']:.2f}", ha='center', va='bottom', fontsize=9)
    ax.text(idx + width/2, row['test_macro_f1'] + 0.015, f"{row['test_macro_f1']:.2f}", ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig_path_pdf = FIG_DIR / 'scicite_direct_input_tow_scores.pdf'
fig_path_png = FIG_DIR / 'scicite_direct_input_tow_scores.png'
fig.savefig(fig_path_pdf, bbox_inches='tight')
fig.savefig(fig_path_png, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', fig_path_pdf)

# Confusion matrix figures.
for condition in order:
    cm_file = RESULTS_DIR / f'{condition}_confusion_matrix.json'
    if not cm_file.exists():
        continue
    cm_data = json.load(open(cm_file, 'r', encoding='utf-8'))
    cm = np.array(cm_data['matrix'])
    labels = cm_data['labels']

    fig, ax = plt.subplots(figsize=(4.4, 4.0))
    im = ax.imshow(cm)
    ax.set_title(condition)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Gold')
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticklabels(labels)
    for ii in range(cm.shape[0]):
        for jj in range(cm.shape[1]):
            ax.text(jj, ii, str(cm[ii, jj]), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    out_pdf = FIG_DIR / f'{condition}_confusion_matrix.pdf'
    out_png = FIG_DIR / f'{condition}_confusion_matrix.png'
    fig.savefig(out_pdf, bbox_inches='tight')
    fig.savefig(out_png, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved confusion matrix:', out_pdf)

In [ ]:
# ============================================================
# 9. Qualitative examples for the paper appendix
# ============================================================
# Save a few representative examples showing what CC-ToW vs IW-ToW look like on the same IDs.
example_rows = []
for split in ['test']:
    raw_map = map_by_id(raw_rows[split])
    a_map = map_by_id(task_rows[split])
    b_map = map_by_id(word_rows[split])
    for ex_id in common_ids[split][:10]:
        r = raw_map[ex_id]
        a = a_map[ex_id]
        b = b_map[ex_id]
        example_rows.append({
            'id': ex_id,
            'label': r.get('label'),
            'label_name': r.get('label_name'),
            'raw_text': r.get('text'),
            'cc_tow_cue_words': a.get('cue_words'),
            'cc_tow_text': a.get('tow'),
            'iw_tow_target_word': b.get('target_word'),
            'iw_tow_category': b.get('tow_category'),
            'iw_tow_augmented_text': b.get('tow_augmented_text'),
        })

examples_path = RESULTS_DIR / 'qualitative_examples_raw_CC_IW.json'
write_json(example_rows, examples_path)
print('Saved qualitative examples:', examples_path)

for ex in example_rows[:3]:
    print('\n' + '=' * 80)
    print('ID:', ex['id'], '| label:', ex['label_name'])
    print('\nRaw:')
    print(ex['raw_text'][:700])
    print('\nCitation-Cue ToW (CC-ToW) ToW:')
    print(ex['cc_tow_text'])
    print('\nInline Word-Level ToW (IW-ToW) target/category:', ex['iw_tow_target_word'], ex['iw_tow_category'])
    print(ex['iw_tow_augmented_text'][:900])

## Output files

After completion, the main outputs are:

```text
/content/drive/MyDrive/tow_project/B/extension/scicite/results_direct_input_tow_extensions/scicite_direct_input_tow_results_paper_table.csv
/content/drive/MyDrive/tow_project/B/extension/scicite/results_direct_input_tow_extensions/scicite_direct_input_tow_results_paper_table.json
/content/drive/MyDrive/tow_project/B/extension/scicite/results_direct_input_tow_extensions/figures/scicite_direct_input_tow_scores.pdf
/content/drive/MyDrive/tow_project/B/extension/scicite/results_direct_input_tow_extensions/qualitative_examples_raw_CC_IW.json
```

This notebook intentionally does not regenerate CC-ToW or IW-ToW data. It only trains and evaluates SciBERT classifiers using cached ToW files.
